# Ch 11 — Baird's Counterexample: Deadly Triad 实际发散

**复现** off-policy linear semi-gradient TD 的 **w 指数发散**——deadly triad 三腿俱全时的 canonical 反例。

## Setup（Sutton 11.2）
- **7 个 state**：state 1-6 + state 7
- **2 个 action**：'dashed'（去 state 1-6 的随机一个） vs 'solid'（必去 state 7）
- **reward 全 0**——所以真实 V_π = 0
- **target policy π** = 总走 solid（必到 state 7）→ V*(s) = 0
- **behavior policy b**：1/7 概率 solid, 6/7 概率 dashed → 大量探索 state 1-6
- **8 个特征**：linear v̂(s, w) = w^T · x(s)，巧妙构造让 state 7 跟 state 1-6 共享参数

## 学什么
1. 即使 V_π=0 完全在 representable space 内
2. semi-gradient TD + linear FA + **off-policy** 仍会 **指数发散**
3. ||w|| 增长到 10^4+——亲眼看 deadly triad 怎么炸

## 链回
- [[ch11-off-policy-methods]] §3.2 Baird
- [[_anki/ch11-deadlytriad-cards]] §B

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)

# 8-dim feature vectors for 7 states (Sutton 11.2 Figure 11.1)
# state 1-6: x[i] = [2 if j == i-1 else 0, ..., 1 in slot 8]
# state 7  : x[6] = [0,0,0,0,0,0,1,2]
FEATURES = np.zeros((7, 8))
for s in range(6):
    FEATURES[s, s] = 2.0
    FEATURES[s, 7] = 1.0
FEATURES[6, 6] = 1.0
FEATURES[6, 7] = 2.0

print('Feature matrix (7 states × 8 features):')
print(FEATURES.astype(int))
print('\nObservation: state 7 shares feature slot 7 with all other states')
print('→ updating w[7] affects ALL state values → instability seed')

## 算法实现

Off-policy semi-gradient TD：
- 每步：用 behavior b 采 (s, a, r, s')
- importance ratio ρ = π(a|s) / b(a|s)
- update: w ← w + α · ρ · δ · x(s)，其中 δ = r + γ · v̂(s') - v̂(s)

In [ ]:
def baird_step(w, s, alpha=0.01, gamma=0.99):
    """Single off-policy semi-gradient TD step from state s."""
    # behavior: 1/7 solid (action 0 → s=6), 6/7 dashed (random of 0..5)
    if np.random.rand() < 1/7:
        a = 'solid'
        s_next = 6  # state 7 (0-indexed = 6)
        rho = 1.0 / (1/7)  # π(solid|s)=1, b(solid|s)=1/7
    else:
        a = 'dashed'
        s_next = np.random.randint(0, 6)
        rho = 0.0 / (6/7)  # π(dashed|s)=0 → ρ = 0
    r = 0  # all rewards 0
    v_s = FEATURES[s] @ w
    v_next = FEATURES[s_next] @ w
    delta = r + gamma * v_next - v_s
    w = w + alpha * rho * delta * FEATURES[s]
    return w, s_next

# Initial w: Sutton uses w = [1, 1, 1, 1, 1, 1, 10, 1]
w = np.array([1, 1, 1, 1, 1, 1, 10, 1], dtype=float)
print(f'Initial w: {w}')
print(f'Initial v̂(s): {FEATURES @ w}')
print(f'True V_π = 0 for all states (target policy always goes to state 7)')

## 跑 1000 步 + 记录 ||w||

In [ ]:
n_steps = 1000
w = np.array([1, 1, 1, 1, 1, 1, 10, 1], dtype=float)
s = np.random.randint(0, 7)
norms = [np.linalg.norm(w)]
w_traces = [w.copy()]
for _ in range(n_steps):
    w, s = baird_step(w, s, alpha=0.01, gamma=0.99)
    norms.append(np.linalg.norm(w))
    w_traces.append(w.copy())

print(f'After {n_steps} steps:')
print(f'  Final w  : {w}')
print(f'  Final ||w||: {norms[-1]:.2f}')
print(f'  v̂(s) after divergence: {FEATURES @ w}')
print(f'  True V_π = 0, but estimates blew up to:', FEATURES @ w)

## 可视化指数发散

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].semilogy(norms)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('||w|| (log scale)')
axes[0].set_title("Baird's counterexample: ||w|| exponential blow-up")
axes[0].grid(alpha=0.3)

w_arr = np.array(w_traces)
for i in range(8):
    axes[1].plot(w_arr[:, i], label=f'w[{i}]')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('w[i] value')
axes[1].set_title('Individual weights diverge')
axes[1].legend(fontsize=8, loc='best')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 关键观察

1. ||w|| 在 1000 步内涨到 **10^3+**（继续跑会到 10^7+）
2. True V = 0，但 v̂(s) 散到几百乃至上千
3. **没有任何外部 reward 推动**——纯粹算法不稳定

## 为什么发散？

**Deadly triad 三腿**：
1. **Function approximation**：linear FA（不是 tabular）
2. **Bootstrapping**：TD target = r + γv̂(s') 用 v̂ 自己
3. **Off-policy**：behavior b ≠ target π，IS ratio ρ 把 update 放大

**机制**：state 7 跟 1-6 共享 w[7]——update state 1-6 的 v̂ 会**同时**改 state 7 的 v̂——target 跟 prediction 同时漂移 → 正反馈炸开。

## 怎么解？

- **GTD / GTD2 / TDC** (Sutton 2009)：真梯度 of MSPBE → 收敛
- **Emphatic-TD** (Sutton 2016)：emphasis weight 抑制不稳更新
- **DQN trick**（NN 而非 linear）：target network + replay → 实际 work（不是真梯度）
- **PPO**：on-policy → 干脆**避开**第 3 条腿
- **DPO**：no RL bootstrap → 三腿一条不沾

## Interview 启示

能讲清 Baird 反例 = 证明懂 deadly triad **真实数学根源**（不是死记三个词）。手写时强调：
- state 7 跟其他 state 共享参数 → update 联动
- IS ratio ρ 放大不稳更新
- semi-gradient ≠ true gradient → 没 loss landscape 保证下降

## 链回
- [[ch11-off-policy-methods]] §3.2 Baird / §3.5 GTD
- [[_anki/ch11-deadlytriad-cards]] Card 6-10